<a href="https://colab.research.google.com/github/stipid/videotools/blob/main/whisperx_dualtrack_aligner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 安装运行环境

## 创建虚拟环境

In [ ]:
!uv venv --clear

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


## *安装* WhisperX

安装依赖

In [ ]:
!uv pip install torchvision==0.23.0
!uv pip install soundfile

Using Python 3.12.13 environment at: /usr
Checked 1 package in 97ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 84ms


In [ ]:
!uv pip install -p .venv/bin/python whisperx

Resolved 115 packages in 158ms
Installed 115 packages in 1.21s
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + alembic==1.18.4
 + antlr4-python3-runtime==4.9.3
 + asteroid-filterbanks==0.4.0
 + attrs==26.1.0
 + av==17.0.0
 + certifi==2026.2.25
 + charset-normalizer==3.4.6
 + click==8.3.1
 + colorlog==6.10.1
 + contourpy==1.3.3
 + ctranslate2==4.7.1
 + cycler==0.12.1
 + einops==0.8.2
 + faster-whisper==1.2.1
 + filelock==3.25.2
 + flatbuffers==25.12.19
 + fonttools==4.62.1
 + frozenlist==1.8.0
 + fsspec==2026.2.0
 + googleapis-common-protos==1.73.0
 + greenlet==3.3.2
 + grpcio==1.78.0
 + hf-xet==1.4.2
 + huggingface-hub==0.36.2
 + idna==3.11
 + importlib-metadata==8.7.1
 + jinja2==3.1.6
 + joblib==1.5.3
 + julius==0.2.7
 + kiwisolver==1.5.0
 + lightning==2.6.1
 + lightning-utilities==0.15.3
 + mako==1.3.10
 + markdown-it-py==4.0.0
 + markupsafe==3.0.3
 + matplotlib==3.10.8
 + mdurl==0.1.2
 + mpmath==1.3.0
 + multidict==6.7.1
 + networkx==3.6.1
 + nltk==3.9.4
 + nump

## 安装/设置 CUDA 支持库

In [ ]:
# !apt install libcudnn8 libcudnn8-dev -y

In [ ]:
# 1. 安装系统 FFmpeg 库 (修复 pyannote-audio 报错)
# !apt-get update && apt-get install -y ffmpeg

In [ ]:
# 1. 安装系统 FFmpeg 库 (修复 pyannote-audio 报错)
# !apt-get update && apt-get install -y ffmpeg libavutil-dev libavcodec-dev libavformat-dev libswscale-dev

# 2. 彻底清除并重新安装 numpy 和 scipy 以解决兼容性问题
# 解决 ImportError: cannot import name '_center' from 'numpy._core.umath'
# !pip install -y numpy scipy
# !pip install --force-reinstall numpy==1.25.2 scipy

# 3. 确保 transformers 库正确安装或更新
# 解决 ModuleNotFoundError: Could not import module 'Wav2Vec2ForCTC'
# !pip install torchvision==0.23.0
# !pip install transformers

# 4. 安装 WhisperX
# !pip install git+https://github.com/m-bain/whisperx.git

In [ ]:
%env MPLBACKEND=agg
%env TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=true
%env LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib

env: MPLBACKEND=agg
env: TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=true
env: LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib


In [ ]:
# 查找 cudnn lib 实际位置
!find / -name "libcudnn*.so*" 2>/dev/null | grep -v proc

/root/.cache/uv/archive-v0/aluP-Z73yaIOuSiUsAT5m/nvidia/cudnn/lib/libcudnn_cnn.so.9
/root/.cache/uv/archive-v0/aluP-Z73yaIOuSiUsAT5m/nvidia/cudnn/lib/libcudnn_graph.so.9
/root/.cache/uv/archive-v0/aluP-Z73yaIOuSiUsAT5m/nvidia/cudnn/lib/libcudnn_adv.so.9
/root/.cache/uv/archive-v0/aluP-Z73yaIOuSiUsAT5m/nvidia/cudnn/lib/libcudnn_heuristic.so.9
/root/.cache/uv/archive-v0/aluP-Z73yaIOuSiUsAT5m/nvidia/cudnn/lib/libcudnn.so.9
/root/.cache/uv/archive-v0/aluP-Z73yaIOuSiUsAT5m/nvidia/cudnn/lib/libcudnn_ops.so.9
/root/.cache/uv/archive-v0/aluP-Z73yaIOuSiUsAT5m/nvidia/cudnn/lib/libcudnn_engines_precompiled.so.9
/root/.cache/uv/archive-v0/aluP-Z73yaIOuSiUsAT5m/nvidia/cudnn/lib/libcudnn_engines_runtime_compiled.so.9
/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn_cnn.so.9
/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn_graph.so.9
/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn_adv.so.9
/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/lib

# 挂载 Google Drive

In [ ]:
from google.colab import drive
import os

# 挂载 Drive
drive.mount('/gdrive')

# --- 配置区 (请根据你的 Drive 实际路径修改) ---
# 建议保持原有结构，例如：
DRIVE_BASE_DIR = '/gdrive/MyDrive/media_proc/speed-align/'
AUDIO_INPUT_DIR = os.path.join(DRIVE_BASE_DIR, 'audio_input') # 存放 mp3/wav
OUTPUT_DIR = os.path.join(DRIVE_BASE_DIR, 'json_output')      # 存放结果 JSON

# 创建输出目录（如果不存在）
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Drive 挂载完成。输入目录: {AUDIO_INPUT_DIR}, 输出目录: {OUTPUT_DIR}")

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
✅ Drive 挂载完成。输入目录: /gdrive/MyDrive/media_proc/speed-align/audio_input, 输出目录: /gdrive/MyDrive/media_proc/speed-align/json_output


# 设置 PDF 原文与文本清洗函数

In [ ]:
import re
import json

# ==========================================
# 🛑🛑🛑在此粘贴你从 PDF 提取的完整英文原文🛑🛑🛑
# ==========================================
MY_FULL_PDF_TEXT = """
Lesson 5,Storm Castle.
Nadim's mum and dad had to go away, so Nadim came to stay at Biff and Chip's house.
Nadim had a bag with all his things, but he had a big box, too.
"What's in that big box?" asked Chip.
"""
# ==========================================

def preprocess_pdf_text(text):
    """清理 PDF 常见的硬换行和多余空格，并按标点符号粗略分段（友好 CPU/GPU 内存）"""
    # 1. 处理 PDF 连词符 (如 "inter-\nnational" -> "international")
    text = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', text)
    # 2. 将硬换行替换为空格
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    # 3. 移除多余的空格
    text = re.sub(r'\s+', ' ', text).strip()

    # 4. 简单的分句逻辑（按 . ! ? 切分），避免单个 Segment 过长
    # 这对于 10 几分钟的长音频对齐稳定性至关重要
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # 过滤掉过短的碎片
    return [s.strip() for s in sentences if len(s.strip()) > 5]

# 执行清洗和分段
clean_segments_to_align = preprocess_pdf_text(MY_FULL_PDF_TEXT)

print(f"✅ PDF 文本已清洗。共切分为 {len(clean_segments_to_align)} 个句子片段用于顺序对齐。")
if len(clean_segments_to_align) > 0:
    print(f"示例前两段:\n1. {clean_segments_to_align[0]}\n2. {clean_segments_to_align[1]}")
else:
    print("⚠️ 警告: PDF 文本为空或切分失败，请检查输入。")

✅ PDF 文本已清洗。共切分为 4 个句子片段用于顺序对齐。
示例前两段:
1. Lesson 5,Storm Castle.
2. Nadim's mum and dad had to go away, so Nadim came to stay at Biff and Chip's house.


# 执行强制对齐主程序

In [ ]:
import sys
sys.path.insert(0, '/content/.venv/lib/python3.12/site-packages')

import whisperx
import torch
import os
import glob
import json
import soundfile as sf   # 用于读取音频时长，pip install soundfile

ALIGNMENT_THRESHOLD = 70.0

# 对齐后端选择：
#   "torchaudio"  → 内存占用小（推荐 Colab CPU），精度略低
#   "whisperx"    → 精度高，但容易 OOM
ALIGN_BACKEND = "torchaudio"

# torchaudio 可用的轻量模型（内存从小到大）：
#   WAV2VEC2_ASR_BASE_960H      ~360MB 推理峰值，精度够用
#   WAV2VEC2_ASR_LARGE_LV60K    ~900MB，精度更好但仍比 whisperx 默认省内存
TORCHAUDIO_MODEL = "WAV2VEC2_ASR_BASE_960H"

# --- 环境检测 (保持不变) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

clean_text_to_align = """
Lesson 5
Storm Castle
Nadim's mum and dad had to go away, so Nadim came to stay at Biff and Chip's house.
Nadim had a bag with all his things, but he had a big box, too.
"What's in that big box?" asked Chip.
"Wait and see," said Nadim.
The children went to Biff's room.
Nadim had the big box. He opened it and took out his computer.
"Brilliant!" said Chip.
"I've got some new computer games," said Nadim. "But we can play with them later. I've got something else to play with."
Wilf came to play. He had one of his new toys. It was a space craft.
Biff, Wilf and Kipper played with all the toys. Kipper said he wanted to be a robot when he grew up.
"You can't," said Biff. "People can't be robots."
"""

# --- 1. 加载对齐模型 ---
model_a, metadata = whisperx.load_align_model(language_code="en", device=device)

# --- 2. 遍历音频文件 ---
# audio_files = []
# for ext in ['*.mp3', '*.wav', '*.m4a']:
#     audio_files.extend(glob.glob(os.path.join(AUDIO_INPUT_DIR, ext)))

# --- 3. 核心对齐循环 + 报告生成 ---
# for audio_path in audio_files:
audio_path = "/gdrive/MyDrive/media_proc/speed-align/audio_input/6b_05_Storm_Castle.mp3"
file_name = os.path.basename(audio_path)
file_basename = os.path.splitext(file_name)[0]
output_json = os.path.join(OUTPUT_DIR, f"{file_basename}_word_timestamps.json")
output_report = os.path.join(OUTPUT_DIR, f"{file_basename}_alignment_report.txt")

print(f"▶️ 正在处理: {file_name}...")

try:
    audio = whisperx.load_audio(audio_path)

    with sf.SoundFile(audio_path) as f:
        duration = len(f) / f.samplerate
    print(f"   音频时长: {duration:.2f} 秒")

    initial_segments = [
            {
                "text":  clean_text_to_align.strip(),
                "start": 0.0,
                "end":   60,   # ✅ 用实际时长，不硬编码 1800.0
            }
        ]

    # ── 步骤 C：强制对齐 ──────────────────────────
    aligned_result = whisperx.align(
        initial_segments,
        model_a,
        metadata,
        audio,
        device,
        return_char_alignments=False,
    )

    # ── 步骤 D：统计对齐结果 ──────────────────────
    total_words     = 0
    aligned_count   = 0
    missing_words   = []
    word_timestamps = []

    for segment in aligned_result["segments"]:
        # ✅ 修复：正确字段名是 "words"，不是 "word_segments"
        for word_info in segment.get("words", []):
            total_words += 1
            start = word_info.get("start")
            end   = word_info.get("end")

            if start is not None and end is not None:
                aligned_count += 1
                word_timestamps.append({
                    "w":     word_info["word"],
                    "s":     round(start, 3),
                    "e":     round(end,   3),
                    "score": round(word_info.get("score", 0.0), 3),
                })
            else:
                missing_words.append(word_info.get("word", "?"))

    alignment_rate = (aligned_count / total_words * 100) if total_words > 0 else 0.0
    status_icon    = "✅" if alignment_rate >= ALIGNMENT_THRESHOLD else "⚠️"

    # ── 步骤 E：生成报告 ──────────────────────────
    report = (
        f"\n{'='*50}\n"
        f"        WHISPERX 对齐实验报告\n"
        f"{'='*50}\n"
        f"音频文件:       {file_name}\n"
        f"音频时长:       {duration:.2f} 秒\n"
        f"状态:           {status_icon} "
        f"{'成功' if alignment_rate >= ALIGNMENT_THRESHOLD else '部分对齐（请检查原文顺序或音频质量）'}\n"
        f"{'─'*50}\n"
        f"原文总词数:     {total_words}\n"
        f"成功对齐词数:   {aligned_count}\n"
        f"未对齐词数:     {total_words - aligned_count}\n"
        f"整体对齐率:     {alignment_rate:.2f}%\n"
        f"{'─'*50}\n"
        f"未对齐词样例（前 20 个）:\n"
        f"  {', '.join(missing_words[:20]) if missing_words else '无（全量对齐成功）'}\n"
        f"{'='*50}\n"
    )
    print(report)

    # ── 步骤 F：保存结果 ──────────────────────────
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(word_timestamps, f, indent=2, ensure_ascii=False)

    with open(output_report, "w", encoding="utf-8") as f:
        f.write(report)

except Exception as e:
    print(f"❌ 处理 {file_name} 时发生错误: {e}")
    # failed_files.append((file_name, str(e)))

# ──────────────────────────────────────────────
# 4. 汇总
# ──────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"🎉 全部处理完成。结果已保存至 {OUTPUT_DIR}/")
# print(f"   成功: {len(audio_files) - len(failed_files)} 个  |  失败: {len(failed_files)} 个")

# if failed_files:
#     print("\n以下文件处理失败：")
#     for name, err in failed_files:
#         print(f"  • {name}: {err}")

print(f"{'='*50}\n")

▶️ 正在处理: 6b_05_Storm_Castle.mp3...
   音频时长: 836.29 秒

        WHISPERX 对齐实验报告
音频文件:       6b_05_Storm_Castle.mp3
音频时长:       836.29 秒
状态:           ✅ 成功
──────────────────────────────────────────────────
原文总词数:     136
成功对齐词数:   136
未对齐词数:     0
整体对齐率:     100.00%
──────────────────────────────────────────────────
未对齐词样例（前 20 个）:
  无（全量对齐成功）


🎉 全部处理完成。结果已保存至 /gdrive/MyDrive/media_proc/speed-align/json_output/



# 整篇对齐

In [ ]:
import sys
sys.path.insert(0, '/content/.venv/lib/python3.12/site-packages')

import re
import json
import gc
import os
import numpy as np
import torch
import whisperx

# ──────────────────────────────────────────────────────────────────
# 配置区
# ──────────────────────────────────────────────────────────────────
AUDIO_PATH   = "/gdrive/MyDrive/media_proc/speed-align/audio_input/6b_05_Storm_Castle.mp3"
OUTPUT_DIR   = "/gdrive/MyDrive/media_proc/speed-align/output"
LANGUAGE     = "en"
WHISPER_MODEL_SIZE = "base"       # Colab CPU 用 base，够快且省内存

# 对齐安全窗口（秒）：单次送入对齐模型的最大音频长度
# 验证 60s 不 OOM，保守起见用 55
SAFE_WINDOW_SEC = 120

# 句子匹配：编辑距离相似度阈值（0~1），低于此值视为未匹配
MATCH_THRESHOLD = 0.35

# 对齐质量分级
TIER_GOOD = 90.0    # 对齐率 > 90%：插值补全未对齐词
TIER_LOW  = 60.0    # 对齐率 60~90%：标记低置信度
                    # 对齐率 < 60%：整句标记失败

# ✅ 原文（带标点的完整字符串）
ORIGINAL_TEXT = """
Lesson 5,Storm Castle.
Nadim's mum and dad had to go away, so Nadim came to stay at Biff and Chip's house.
Nadim had a bag with all his things, but he had a big box, too.
"What's in that big box?" asked Chip.
"Wait and see," said Nadim.
The children went to Biff's room.
Nadim had the big box. He opened it and took out his computer.
"Brilliant!" said Chip.
"I've got some new computer games," said Nadim. "But we can play with them later. I've got something else to play with."
Nadim had some robots.
"These are great," said Kipper.
"I'm going to get one like this. It's my favourite."
Biff and Chip had a robot, too. They went to fetch it.
"We're going to have a great time," said Nadim.
Wilf came to play. He had one of his new toys. It was a space craft.
Biff, Wilf and Kipper played with all the toys. Kipper said he wanted to be a robot when he grew up.
"You can't," said Biff. "People can't be robots."
Chip and Nadim played on the computer. One of Nadim's new games was called "Storm Castle". It looked exciting.
"It's quite hard," said Nadim. "I'll show you how to play, then you can have a go."
"It looks great," said Chip.
"""

# ──────────────────────────────────────────────────────────────────
# 工具函数
# ──────────────────────────────────────────────────────────────────

def split_sentences(text: str) -> list[str]:
    """
    按标点切句，保留引号内的完整句子。
    返回非空句子列表。
    """
    # 在句末标点后切分，保留标点
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = []
    for p in parts:
        p = p.strip()
        if p:
            sentences.append(p)
    return sentences


def normalize(text: str) -> list[str]:
    """文本标准化：小写、去标点，返回词列表。"""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    return text.split()


def word_edit_distance(words_a: list[str], words_b: list[str]) -> int:
    """词级编辑距离（动态规划）。"""
    m, n = len(words_a), len(words_b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, n + 1):
            if words_a[i-1] == words_b[j-1]:
                dp[j] = prev[j-1]
            else:
                dp[j] = 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]


def similarity(text_a: str, text_b: str) -> float:
    """两段文本的词级相似度（0~1）。"""
    wa, wb = normalize(text_a), normalize(text_b)
    if not wa or not wb:
        return 0.0
    dist = word_edit_distance(wa, wb)
    return 1.0 - dist / max(len(wa), len(wb))


def match_sentences_to_segments(
    sentences: list[str],
    whisper_segments: list[dict],
    threshold: float,
) -> list[dict]:
    """
    为每个原文句子找到最匹配的 Whisper segment，
    返回每句的 {sentence, start, end, matched, sim} 列表。

    策略：
    - 每句在转写 segments 里滑动窗口搜索（1~3 个 segment 合并）
    - 找不到匹配（sim < threshold）→ 向相邻句子借时间
    """
    n_seg = len(whisper_segments)
    results = []

    # 把 whisper segments 拼成候选窗口（1句、2句、3句合并）
    def get_window_text(i, j):
        return " ".join(s["text"] for s in whisper_segments[i:j+1])

    def get_window_time(i, j):
        return whisper_segments[i]["start"], whisper_segments[j]["end"]

    seg_cursor = 0  # 从上次匹配位置往后搜索，避免时间倒退

    for sent in sentences:
        best_sim   = -1.0
        best_start = None
        best_end   = None
        best_i     = seg_cursor

        # 搜索范围：从 seg_cursor 往后最多 10 个 segment
        search_end = min(seg_cursor + 10, n_seg)
        for i in range(seg_cursor, search_end):
            for window in range(1, 4):  # 1~3 个 segment 合并
                j = i + window - 1
                if j >= n_seg:
                    break
                candidate = get_window_text(i, j)
                sim = similarity(sent, candidate)
                if sim > best_sim:
                    best_sim   = sim
                    best_start, best_end = get_window_time(i, j)
                    best_i     = i

        matched = best_sim >= threshold
        if matched:
            seg_cursor = best_i + 1  # 下次从这之后开始搜

        results.append({
            "sentence": sent,
            "start":    best_start,
            "end":      best_end,
            "matched":  matched,
            "sim":      round(best_sim, 3),
        })

    # ── 填补未匹配句子的时间窗口 ──────────────────────────
    # 向前/后已匹配的句子借时间，线性插值边界
    n = len(results)
    for i, r in enumerate(results):
        if r["matched"]:
            continue
        # 找前一个已匹配
        prev_end = results[i-1]["end"] if i > 0 and results[i-1]["end"] is not None else 0.0
        # 找后一个已匹配
        next_start = None
        for j in range(i+1, n):
            if results[j]["matched"] and results[j]["start"] is not None:
                next_start = results[j]["start"]
                break
        if next_start is None:
            next_start = prev_end + 30.0  # 兜底：给 30 秒窗口

        # 按词数比例在 [prev_end, next_start] 内分配时间
        unmatched_block = [results[k] for k in range(i, n) if not results[k]["matched"]]
        total_words = sum(len(normalize(r["sentence"])) for r in unmatched_block)
        cursor = prev_end
        for r2 in unmatched_block:
            w = len(normalize(r2["sentence"]))
            frac = w / total_words if total_words > 0 else 1.0 / len(unmatched_block)
            r2["start"] = round(cursor, 3)
            r2["end"]   = round(cursor + (next_start - prev_end) * frac, 3)
            cursor      = r2["end"]
        break  # 已经批量处理了连续未匹配块，外层循环继续

    return results


def interpolate_missing_words(
    word_timestamps: list[dict],
    missing_words: list[str],
    seg_start: float,
    seg_end: float,
) -> list[dict]:
    """
    对未对齐的词做线性插值：
    在已对齐词的时间缝隙中，按词数均匀分配时间。
    """
    if not missing_words:
        return word_timestamps

    # 找最后一个已对齐词的结束时间
    last_end = word_timestamps[-1]["e"] if word_timestamps else seg_start
    remaining = seg_end - last_end
    step = remaining / (len(missing_words) + 1)

    for idx, word in enumerate(missing_words):
        word_timestamps.append({
            "w":          word,
            "s":          round(last_end + step * (idx + 1), 3),
            "e":          round(last_end + step * (idx + 2), 3),
            "score":      0.0,
            "confidence": "interpolated",
        })
    return word_timestamps


def align_sentence(
    sentence: str,
    audio: np.ndarray,
    seg_start: float,
    seg_end: float,
    align_model,
    align_metadata,
    device: str,
    sample_rate: int = 16000,
) -> tuple[list[dict], list[str], float]:
    """
    对单句执行强制对齐。
    返回：(word_timestamps, missing_words, alignment_rate)
    时间戳已加上 seg_start 偏移。
    """
    # 裁剪音频片段（控制在 SAFE_WINDOW_SEC 内）
    start_sample = int(seg_start * sample_rate)
    end_sample   = int(min(seg_end, seg_start + SAFE_WINDOW_SEC) * sample_rate)
    end_sample   = min(end_sample, len(audio))
    audio_chunk  = audio[start_sample:end_sample]
    chunk_dur    = len(audio_chunk) / sample_rate

    segments_input = [{
        "text":  sentence.strip(),
        "start": 0.0,
        "end":   chunk_dur,
    }]

    try:
        result = whisperx.align(
            segments_input,
            align_model,
            align_metadata,
            audio_chunk,
            device,
            return_char_alignments=False,
        )
    except Exception as e:
        print(f"     ⚠️  align() 异常: {e}")
        return [], normalize(sentence), 0.0
    finally:
        del audio_chunk
        gc.collect()

    word_timestamps = []
    missing_words   = []
    total = 0

    for seg in result["segments"]:
        for w in seg.get("words", []):
            total += 1
            s = w.get("start")
            e = w.get("end")
            if s is not None and e is not None:
                word_timestamps.append({
                    "w":          w["word"],
                    "s":          round(s + seg_start, 3),
                    "e":          round(e + seg_start, 3),
                    "score":      round(w.get("score", 0.0), 3),
                    "confidence": "high",
                })
            else:
                missing_words.append(w.get("word", "?"))

    rate = (len(word_timestamps) / total * 100) if total > 0 else 0.0
    return word_timestamps, missing_words, rate


# ──────────────────────────────────────────────────────────────────
# 主流程
# ──────────────────────────────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device       = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "float32"
    file_basename = os.path.splitext(os.path.basename(AUDIO_PATH))[0]

    print(f"🖥️  设备: {device.upper()} | 精度: {compute_type}")
    print(f"📄 原文句子切分中...")

    # ── 步骤 1：切句 ──────────────────────────────────────────────
    sentences = split_sentences(ORIGINAL_TEXT)
    print(f"   共 {len(sentences)} 句")
    for i, s in enumerate(sentences):
        print(f"   [{i+1}] {s[:60]}{'...' if len(s)>60 else ''}")

    # ── 步骤 2：Whisper 粗转录 ────────────────────────────────────
    print(f"\n⏳ Whisper 粗转录 (model={WHISPER_MODEL_SIZE})...")
    whisper_model = whisperx.load_model(
        WHISPER_MODEL_SIZE,
        device=device,
        compute_type=compute_type,
        language=LANGUAGE,
    )
    audio = whisperx.load_audio(AUDIO_PATH)
    transcription = whisper_model.transcribe(audio, batch_size=4)
    whisper_segments = transcription["segments"]

    # 新增：将 Whisper 粗转录结果保存为 JSON
    whisper_segments_json_path = os.path.join(OUTPUT_DIR, f"{file_basename}_whisper_segments.json")
    with open(whisper_segments_json_path, "w", encoding="utf-8") as f:
        json.dump(whisper_segments, f, indent=2, ensure_ascii=False)
    print(f"   Whisper 粗转录结果已保存至: {whisper_segments_json_path}")

    print(f"   转写得到 {len(whisper_segments)} 个 segment")

    # 释放 Whisper 模型（节省内存给对齐模型）
    del whisper_model
    gc.collect()

    # ── 步骤 3：原文句子匹配转写 segment ─────────────────────────
    print(f"\n🔗 原文句子 → 转写 segment 匹配中...")
    matched = match_sentences_to_segments(sentences, whisper_segments, MATCH_THRESHOLD)
    unmatched_count = sum(1 for m in matched if not m["matched"])
    print(f"   匹配成功: {len(matched)-unmatched_count}/{len(matched)} 句"
          f"  未匹配（已插值时间）: {unmatched_count} 句")
    for m in matched:
        icon = "✅" if m["matched"] else "⚠️ "
        print(f"   {icon} [{m['start']:.1f}s~{m['end']:.1f}s] sim={m['sim']:.2f}  {m['sentence'][:50]}")

    # ── 步骤 4：加载对齐模型 ──────────────────────────────────────
    print(f"\n⏳ 加载对齐模型...")
    align_model, align_metadata = whisperx.load_align_model(
        language_code=LANGUAGE,
        device=device,
    )
    gc.collect()

    # ── 步骤 5：逐句强制对齐 ──────────────────────────────────────
    print(f"\n🎯 逐句强制对齐中...\n{'─'*50}")
    all_words         = []
    sentence_reports  = []

    for idx, m in enumerate(matched):
        sent      = m["sentence"]
        seg_start = m["start"] or 0.0
        seg_end   = m["end"]   or (seg_start + SAFE_WINDOW_SEC)

        print(f"  [{idx+1}/{len(matched)}] {sent[:50]}...")
        print(f"         时间窗口: {seg_start:.4f}s ~ {seg_end:.4f}s")

        word_ts, missing, rate = align_sentence(
            sent, audio, seg_start, seg_end,
            align_model, align_metadata, device,
        )

        # 按对齐率分级处理
        if rate >= TIER_GOOD:
            # 插值补全未对齐词
            word_ts = interpolate_missing_words(word_ts, missing, seg_start, seg_end)
            status = "✅ 成功"
        elif rate >= TIER_LOW:
            # 标记低置信度，不插值
            for w in word_ts:
                w["confidence"] = "low"
            status = "⚠️  低置信度"
        else:
            status = "❌ 失败（整句标记）"

        print(f"         对齐率: {rate:.1f}%  {status}  未对齐词: {len(missing)}")

        all_words.extend(word_ts)
        sentence_reports.append({
            "sentence":      sent,
            "start":         seg_start,
            "end":           seg_end,
            "alignment_rate": round(rate, 2),
            "status":        status,
            "missing_words": missing,
            "matched_segment": m["matched"],
            "match_sim":     m["sim"],
        })

    # ── 步骤 6：输出 ──────────────────────────────────────────────
    output_json   = os.path.join(OUTPUT_DIR, f"{file_basename}_word_timestamps.json")
    output_report = os.path.join(OUTPUT_DIR, f"{file_basename}_alignment_report.txt")

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(all_words, f, indent=2, ensure_ascii=False)

    total_words   = len(all_words)
    high_conf     = sum(1 for w in all_words if w.get("confidence") == "high")
    interpolated  = sum(1 for w in all_words if w.get("confidence") == "interpolated")
    low_conf      = sum(1 for w in all_words if w.get("confidence") == "low")
    overall_rate  = (high_conf / total_words * 100) if total_words > 0 else 0.0

    report_lines = [
        f"\n{'='*55}",
        f"         WHISPERX 完整对齐报告",
        f"{'='*55}",
        f"音频文件:        {os.path.basename(AUDIO_PATH)}",
        f"原文句子数:      {len(sentences)}",
        f"{'─'*55}",
        f"总词数:          {total_words}",
        f"高置信度词:      {high_conf}  ({high_conf/total_words*100:.1f}%)",
        f"低置信度词:      {low_conf}  ({low_conf/total_words*100:.1f}%)",
        f"插值补全词:      {interpolated}  ({interpolated/total_words*100:.1f}%)",
        f"整体对齐率:      {overall_rate:.2f}%",
        f"{'─'*55}",
        f"逐句报告:",
    ]
    for i, sr in enumerate(sentence_reports):
        report_lines.append(
            f"  [{i+1:02d}] {sr['alignment_rate']:5.1f}%  {sr['status']}  "
            f"[{sr['start']:.1f}s~{sr['end']:.1f}s]  "
            f"{sr['sentence'][:45]}"
        )
        if sr["missing_words"]:
            report_lines.append(
                f"        未对齐: {', '.join(sr['missing_words'][:10])}"
            )
    report_lines.append(f"{'='*55}\n")
    report = "\n".join(report_lines)

    print(report)
    with open(output_report, "w", encoding="utf-8") as f:
        f.write(report)

    print(f"💾 JSON  → {output_json}")
    print(f"💾 报告  → {output_report}")
    print(f"\n🎉 完成！")

In [ ]:
main()

🖥️  设备: CUDA | 精度: float16
📄 原文句子切分中...
   共 28 句
   [1] Lesson 5,Storm Castle.
   [2] Nadim's mum and dad had to go away, so Nadim came to stay at...
   [3] Nadim had a bag with all his things, but he had a big box, t...
   [4] "What's in that big box?" asked Chip.
   [5] "Wait and see," said Nadim.
   [6] The children went to Biff's room.
   [7] Nadim had the big box.
   [8] He opened it and took out his computer.
   [9] "Brilliant!" said Chip.
   [10] "I've got some new computer games," said Nadim.
   [11] "But we can play with them later.
   [12] I've got something else to play with."
Nadim had some robots...
   [13] "These are great," said Kipper.
   [14] "I'm going to get one like this.
   [15] It's my favourite."
Biff and Chip had a robot, too.
   [16] They went to fetch it.
   [17] "We're going to have a great time," said Nadim.
   [18] Wilf came to play.
   [19] He had one of his new toys.
   [20] It was a space craft.
   [21] Biff, Wilf and Kipper played with all the toys.
  

INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint .venv/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint .venv/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


   Whisper 粗转录结果已保存至: /gdrive/MyDrive/media_proc/speed-align/output/6b_05_Storm_Castle_whisper_segments.json
   转写得到 30 个 segment

🔗 原文句子 → 转写 segment 匹配中...
   匹配成功: 1/28 句  未匹配（已插值时间）: 27 句
   ⚠️  [0.0s~0.0s] sim=0.09  Lesson 5,Storm Castle.
   ✅ [0.4s~30.3s] sim=0.53  Nadim's mum and dad had to go away, so Nadim came 
   ⚠️  [0.0s~0.0s] sim=0.13  Nadim had a bag with all his things, but he had a 
   ⚠️  [0.0s~0.0s] sim=0.18  "What's in that big box?" asked Chip.
   ⚠️  [0.0s~0.1s] sim=0.10  "Wait and see," said Nadim.
   ⚠️  [0.1s~0.1s] sim=0.16  The children went to Biff's room.
   ⚠️  [0.1s~0.1s] sim=0.10  Nadim had the big box.
   ⚠️  [0.1s~0.1s] sim=0.21  He opened it and took out his computer.
   ⚠️  [0.1s~0.1s] sim=0.06  "Brilliant!" said Chip.
   ⚠️  [0.1s~0.1s] sim=0.14  "I've got some new computer games," said Nadim.
   ⚠️  [0.1s~0.1s] sim=0.14  "But we can play with them later.
   ⚠️  [0.1s~0.1s] sim=0.20  I've got something else to play with."
Nadim had s
   ⚠️  [0.1s~0.2

# 优化匹配算法

In [ ]:
import sys
sys.path.insert(0, '/content/.venv/lib/python3.12/site-packages')

import re
import json
import gc
import os
import numpy as np
import torch
import whisperx
import difflib
import time

# ──────────────────────────────────────────────────────────────────
# 配置区
# ──────────────────────────────────────────────────────────────────
AUDIO_PATH   = "/gdrive/MyDrive/media_proc/speed-align/audio_input/6b_05_Storm_Castle.mp3"
WHISPER_JSON_PATH = "/gdrive/MyDrive/media_proc/speed-align/output/6b_05_Storm_Castle_whisper_segments.json"
OUTPUT_DIR   = "/gdrive/MyDrive/media_proc/speed-align/output"
LANGUAGE     = "en"

# 强制对齐分块长度（秒），防止 OOM
CHUNK_MAX_DURATION = 60.0
# 音频切片余量（秒），防止首尾词切断
CHUNK_PADDING = 1.5

# ✅ 原文
ORIGINAL_TEXT = """
Lesson Five.
Storm Castle.
Nadim's mum and dad had to go away, so Nadim came to stay at Biff and Chip's house.
Nadim had a bag with all his things, but he had a big box, too.
"What's in that big box?" asked Chip.
"Wait and see," said Nadim.
The children went to Biff's room.
Nadim had the big box. He opened it and took out his computer.
"Brilliant!" said Chip.
"I've got some new computer games," said Nadim. "But we can play with them later. I've got something else to play with."
Nadim had some robots.
"These are great," said Kipper.
"I'm going to get one like this. It's my favourite."
Biff and Chip had a robot, too. They went to fetch it.
"We're going to have a great time," said Nadim.
Wilf came to play. He had one of his new toys. It was a space craft.
Biff, Wilf and Kipper played with all the toys. Kipper said he wanted to be a robot when he grew up.
"You can't," said Biff. "People can't be robots."
Chip and Nadim played on the computer. One of Nadim's new games was called "Storm Castle". It looked exciting.
"It's quite hard," said Nadim. "I'll show you how to play, then you can have a go."
"It looks great," said Chip.
"""

# ──────────────────────────────────────────────────────────────────
# 工具函数
# ──────────────────────────────────────────────────────────────────

def split_sentences(text: str) -> list[str]:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]

def normalize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    return text.split()

def align_chunk(text, audio, start_t, end_t, align_model, align_metadata, device):
    sr = 16000
    s_idx, e_idx = int(start_t * sr), int(end_t * sr)
    chunk = audio[s_idx:e_idx]
    segments = [{"text": text, "start": 0.0, "end": len(chunk)/sr}]

    try:
        result = whisperx.align(segments, align_model, align_metadata, chunk, device)
        words_out = []
        for s in result["segments"]:
            for w in s.get("words", []):
                if "start" in w:
                    words_out.append({
                        "w": w["word"],
                        "s": round(w["start"] + start_t, 3),
                        "e": round(w["end"] + start_t, 3),
                        "score": w.get("score", 0)
                    })
        return words_out
    except Exception as e:
        print(f"      ❌ [Align Error]: {e}")
        return []

def clean_for_align(t):
    # 在逗号、句号等标点后增加空格，并确保单词间只有单空格
    t = re.sub(r'([,.!?])', r'\1 ', t)
    return " ".join(t.split())

# ──────────────────────────────────────────────────────────────────
# 主流程
# ──────────────────────────────────────────────────────────────────
def main():
    start_time_all = time.time()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    file_basename = os.path.splitext(os.path.basename(AUDIO_PATH))[0]

    print(f"🚀 [开始任务] 设备: {device.upper()}")

    # 1. 切句
    sentences = split_sentences(ORIGINAL_TEXT)
    print(f"📄 [步骤 1/6] 原文切分完成，共 {len(sentences)} 句。")
    for i, s in enumerate(sentences[:3]):
        print(f"   - 示例 {i+1}: {s[:50]}...")

    # 2. 数据加载
    print(f"📂 [步骤 2/6] 正在读取 Whisper JSON: {os.path.basename(WHISPER_JSON_PATH)}")
    with open(WHISPER_JSON_PATH, "r", encoding="utf-8") as f:
        whisper_segs = json.load(f)
    print(f"   ✅ 加载了 {len(whisper_segs)} 个转录片段。")

    print(f"🎵 [步骤 2/6] 正在加载音频文件...")
    audio = whisperx.load_audio(AUDIO_PATH)
    audio_dur = len(audio) / 16000
    print(f"   ✅ 音频加载成功，总时长: {audio_dur:.2f}s")

    # 3. 全局映射
    print(f"🔗 [步骤 3/6] 开始全局序列比对 (difflib)...")
    ref_words = []
    for s in whisper_segs:
        for w in normalize(s["text"]):
            ref_words.append({"word": w, "start": s["start"], "end": s["end"]})

    origin_words_list = []
    word_to_sent_map = []
    for i, sent in enumerate(sentences):
        for w in normalize(sent):
            origin_words_list.append(w)
            word_to_sent_map.append(i)

    print(f"   - 原文总词数: {len(origin_words_list)}")
    print(f"   - Whisper 转录总词数: {len(ref_words)}")

    matcher = difflib.SequenceMatcher(None, origin_words_list, [r["word"] for r in ref_words])
    sent_times = [{"start": None, "end": None} for _ in range(len(sentences))]
    match_count = 0

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            match_count += (i2 - i1)
            for idx in range(i1, i2):
                ref_idx = j1 + (idx - i1)
                s_idx = word_to_sent_map[idx]
                t_s, t_e = ref_words[ref_idx]["start"], ref_words[ref_idx]["end"]
                if sent_times[s_idx]["start"] is None or t_s < sent_times[s_idx]["start"]:
                    sent_times[s_idx]["start"] = t_s
                if sent_times[s_idx]["end"] is None or t_e > sent_times[s_idx]["end"]:
                    sent_times[s_idx]["end"] = t_e

    print(f"   ✅ 比对完成。匹配上的词数: {match_count} ({match_count/len(origin_words_list)*100:.1f}%)")

    # 4. 线性插值
    print(f"📏 [步骤 4/6] 正在对未匹配句子进行时间插值补全...")
    last_known_end = 0.0
    interpolated_count = 0
    for i in range(len(sent_times)):
        if sent_times[i]["start"] is None:
            interpolated_count += 1
            next_start = audio_dur
            for j in range(i + 1, len(sent_times)):
                if sent_times[j]["start"] is not None:
                    next_start = sent_times[j]["start"]
                    break
            sent_times[i]["start"] = last_known_end
            sent_times[i]["end"] = (last_known_end + next_start) / 2
        last_known_end = sent_times[i]["end"]
    print(f"   ✅ 插值完成，共补全 {interpolated_count} 个句子的粗略时间。")

    # 5. 强制对齐
    print(f"⏳ [步骤 5/6] 加载 WhisperX 对齐模型 (Language: {LANGUAGE})...")
    align_model, align_metadata = whisperx.load_align_model(language_code=LANGUAGE, device=device, model_name="WAV2VEC2_ASR_LARGE_LV60K_960H")

    all_word_timestamps = []
    current_chunk_sents = []
    chunk_start_time = None

    print(f"\n🎯 [开始分块强制对齐] (Max Chunk: {CHUNK_MAX_DURATION}s, Padding: {CHUNK_PADDING}s)")
    print("-" * 80)

    for i in range(len(sentences)):
        if chunk_start_time is None: chunk_start_time = sent_times[i]["start"]
        current_chunk_sents.append(i)

        is_last = (i == len(sentences) - 1)
        # 触发对齐条件：达到时长上限 或 是最后一句
        if (sent_times[i]["end"] - chunk_start_time > CHUNK_MAX_DURATION) or is_last:
            # combined_text = " ".join([sentences[idx] for idx in current_chunk_sents])
            combined_text = " ".join([clean_for_align(sentences[idx]) for idx in current_chunk_sents])

            # 计算带 Padding 的安全窗口
            safe_start = max(0.0, chunk_start_time - CHUNK_PADDING)
            safe_end = min(audio_dur, sent_times[i]["end"] + CHUNK_PADDING)

            print(f"📦 处理 Chunk (第 {len(all_word_timestamps)+1} 组词):")
            print(combined_text)
            print(f"   - 句子索引: {current_chunk_sents[0]} 到 {current_chunk_sents[-1]}")
            print(f"   - 预估区间: {chunk_start_time:.2f}s -> {sent_times[i]['end']:.2f}s")
            print(f"   - 实际裁剪: {safe_start:.2f}s -> {safe_end:.2f}s (时长: {safe_end-safe_start:.2f}s)")

            chunk_words = align_chunk(combined_text, audio, safe_start, safe_end, align_model, align_metadata, device)

            print(f"   - 对齐结果: 成功捕获 {len(chunk_words)} 个单词")
            all_word_timestamps.extend(chunk_words)
            print("-" * 40)

            current_chunk_sents = []
            chunk_start_time = None

    # 6. 保存
    print(f"\n💾 [步骤 6/6] 正在导出结果...")
    output_json = os.path.join(OUTPUT_DIR, f"{file_basename}_final_word_ts.json")
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(all_word_timestamps, f, indent=2, ensure_ascii=False)

    end_time_all = time.time()
    print(f"\n🎉 [任务完成!]")
    print(f"   - 总耗时: {end_time_all - start_time_all:.2f} 秒")
    print(f"   - 最终词级时间戳文件: {output_json}")
    print(f"   - 总计单词数: {len(all_word_timestamps)}")

In [ ]:
main()

🚀 [开始任务] 设备: CPU
📄 [步骤 1/6] 原文切分完成，共 29 句。
   - 示例 1: Lesson Five....
   - 示例 2: Storm Castle....
   - 示例 3: Nadim's mum and dad had to go away, so Nadim came ...
📂 [步骤 2/6] 正在读取 Whisper JSON: 6b_05_Storm_Castle_whisper_segments.json
   ✅ 加载了 30 个转录片段。
🎵 [步骤 2/6] 正在加载音频文件...
   ✅ 音频加载成功，总时长: 832.37s
🔗 [步骤 3/6] 开始全局序列比对 (difflib)...
   - 原文总词数: 219
   - Whisper 转录总词数: 1355
   ✅ 比对完成。匹配上的词数: 205 (93.6%)
📏 [步骤 4/6] 正在对未匹配句子进行时间插值补全...
   ✅ 插值完成，共补全 0 个句子的粗略时间。
⏳ [步骤 5/6] 加载 WhisperX 对齐模型 (Language: en)...

🎯 [开始分块强制对齐] (Max Chunk: 60.0s, Padding: 1.5s)
--------------------------------------------------------------------------------
📦 处理 Chunk (第 1 组词):
Lesson Five. Storm Castle. Nadim's mum and dad had to go away, so Nadim came to stay at Biff and Chip's house. Nadim had a bag with all his things, but he had a big box, too. "What's in that big box? " asked Chip. "Wait and see, " said Nadim. The children went to Biff's room. Nadim had the big box. He opened it and took out his computer. "B

代码重构版本

In [ ]:
import whisperx
# ──────────────────────────────────────────────────────────────────
# 2. 工具函数 (保留核心处理逻辑)
# ──────────────────────────────────────────────────────────────────

def split_sentences(text: str) -> list[str]:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]

def normalize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    return text.split()

def clean_for_align(t: str) -> str:
    t = re.sub(r'([,.!?])', r'\1 ', t)
    return " ".join(t.split())

def align_chunk(text, audio, start_t, end_t, align_model, align_metadata, device):
    sr = 16000
    s_idx, e_idx = int(start_t * sr), int(end_t * sr)
    chunk = audio[s_idx:e_idx]
    segments = [{"text": text, "start": 0.0, "end": len(chunk)/sr}]
    try:
        result = whisperx.align(segments, align_model, align_metadata, chunk, device)
        words_out = []
        for s in result["segments"]:
            for w in s.get("words", []):
                if "start" in w:
                    words_out.append({
                        "w": w["word"],
                        "s": round(w["start"] + start_t, 3),
                        "e": round(w["end"] + start_t, 3),
                        "score": w.get("score", 0)
                    })
        return words_out
    except Exception as e:
        print(f"      ❌ [Align Error]: {e}")
        return []

In [ ]:
import sys
sys.path.insert(0, '/content/.venv/lib/python3.12/site-packages')

import os
import re
import json
import time
import torch
import difflib
import whisperx

# ──────────────────────────────────────────────────────────────────
# 主处理函数
# ──────────────────────────────────────────────────────────────────

def run_alignment(
    original_text,
    audio_path,
    whisper_json_path,
    output_dir,
    language="en",
    chunk_max_duration=60.0,
    chunk_padding=1.5
):
    start_time_all = time.time()
    os.makedirs(output_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    file_basename = os.path.splitext(os.path.basename(audio_path))[0]

    print(f"🚀 [开始任务] 设备: {device.upper()}")

    audio = whisperx.load_audio(audio_path)
    audio_dur = len(audio) / 16000
    print(f"   ✅ 数据加载成功，音频时长: {audio_dur:.2f}s")

    # 1. 切句
    sentences = split_sentences(original_text)
    print(f"📄 [步骤 1/6] 原文切分完成，共 {len(sentences)} 句。")

    # 2. 数据加载
    with open(whisper_json_path, "r", encoding="utf-8") as f:
        whisper_segs = json.load(f)


    # 3. 全局映射 (difflib)
    print(f"🔗 [步骤 3/6] 开始全局序列比对...")
    ref_words = []
    for s in whisper_segs:
        for w in normalize(s["text"]):
            ref_words.append({"word": w, "start": s["start"], "end": s["end"]})

    origin_words_list, word_to_sent_map = [], []
    for i, sent in enumerate(sentences):
        normalized_words = normalize(sent)
        for w in normalized_words:
            origin_words_list.append(w)
            word_to_sent_map.append(i)

    matcher = difflib.SequenceMatcher(None, origin_words_list, [r["word"] for r in ref_words])
    sent_times = [{"start": None, "end": None} for _ in range(len(sentences))]

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            for idx in range(i1, i2):
                ref = ref_words[j1 + (idx - i1)]
                s_idx = word_to_sent_map[idx]
                if sent_times[s_idx]["start"] is None or ref["start"] < sent_times[s_idx]["start"]:
                    sent_times[s_idx]["start"] = ref["start"]
                if sent_times[s_idx]["end"] is None or ref["end"] > sent_times[s_idx]["end"]:
                    sent_times[s_idx]["end"] = ref["end"]

    # 4. 线性插值
    print(f"📏 [步骤 4/6] 正在对未匹配句子进行时间插值补全...")
    last_known_end = 0.0
    for i in range(len(sent_times)):
        if sent_times[i]["start"] is None:
            next_start = next((sent_times[j]["start"] for j in range(i+1, len(sent_times)) if sent_times[j]["start"]), audio_dur)
            sent_times[i]["start"] = last_known_end
            sent_times[i]["end"] = (last_known_end + next_start) / 2
        last_known_end = sent_times[i]["end"]

    # 5. 强制对齐
    print(f"⏳ [步骤 5/6] 加载 WhisperX 模型执行对齐...")
    align_model, align_metadata = whisperx.load_align_model(language_code=language, device=device)

    all_word_timestamps = []
    current_chunk_sents, chunk_start_time = [], None

    print(f"\n🎯 [开始分块强制对齐] (Max Chunk: {chunk_max_duration}s)")
    print("-" * 80)

    for i in range(len(sentences)):
        if chunk_start_time is None: chunk_start_time = sent_times[i]["start"]
        current_chunk_sents.append(i)

        if (sent_times[i]["end"] - chunk_start_time > chunk_max_duration) or (i == len(sentences) - 1):
            combined_text = " ".join([clean_for_align(sentences[idx]) for idx in current_chunk_sents])
            safe_start = max(0.0, chunk_start_time - chunk_padding)
            safe_end = min(audio_dur, sent_times[i]["end"] + chunk_padding)

            print(f"📦 处理 Chunk (索引 {current_chunk_sents[0]} 到 {current_chunk_sents[-1]}):")
            chunk_words = align_chunk(combined_text, audio, safe_start, safe_end, align_model, align_metadata, device)
            all_word_timestamps.extend(chunk_words)
            print(f"   - 成功捕获 {len(chunk_words)} 个单词\n" + "-" * 40)

            current_chunk_sents, chunk_start_time = [], None

    # ──────────────────────────────────────────────────────────────────
    # 🌟 核心新增：将 all_word_timestamps 重新封装为标准 segments
    # 🌟 核心修正：增加对齐失败单词的兼容处理
    # ──────────────────────────────────────────────────────────────────
    print(f"🧩 [步骤 6/6] 正在组装 WhisperX 标准 segments 结构...")
    final_segments = []
    word_cursor = 0
    total_aligned = len(all_word_timestamps)

    for i, sent_text in enumerate(sentences):
        # 计算当前句子在原文中应有的单词数
        sent_tokens = normalize(sent_text)
        sent_word_count = len(sent_tokens)
        if sent_word_count == 0: continue

        # 从扁平列表中提取对应数量的词
        segment_words = []
        for _ in range(sent_word_count):
            if word_cursor < total_aligned:
                word_data = all_word_timestamps[word_cursor]
                # 🛡️ 兼容处理：检查是否存在 start 键
                if "start" in word_data:
                    segment_words.append(word_data)
                else:
                    # 如果该词没有时间戳，打印一条警告，但不中断程序
                    print(f"      ⚠️ [跳过无效单词]: 句子 {i} 中的 '{word_data.get('word', 'unknown')}' 对齐失败，无时间戳。")
                word_cursor += 1

        # 🛡️ 健壮性检查：确保 segment_words 不为空且首尾词都有时间戳
        if segment_words:
            try:
                final_segments.append({
                    "start": segment_words[0]["start"],
                    "end": segment_words[-1]["end"],
                    "text": sent_text,
                    "words": segment_words
                })
            except KeyError as e:
                print(f"      ❌ [Segment 错误]: 句子 {i} 存在无法处理的单词数据: {e}")
        else:
            print(f"      ⚠️ [句子对齐失败]: 第 {i} 句未捕获到任何有效词级时间戳，已跳过。")

    # ──────────────────────────────────────────────────────────────────
    # 6. 保存
    # ──────────────────────────────────────────────────────────────────
    output_data = {"segments": final_segments}
    output_json = os.path.join(output_dir, f"{file_basename}_final_segments.json")
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)

    print(f"\n🎉 [任务完成!] 总耗时: {time.time() - start_time_all:.2f}s")
    print(f"   - 最终文件: {output_json}")

    return output_data

In [ ]:
ORIGINAL_TEXT = """
Lesson Five.
Storm Castle.
Nadim's mum and dad had to go away, so Nadim came to stay at Biff and Chip's house.
Nadim had a bag with all his things, but he had a big box, too.
"What's in that big box?" asked Chip.
"Wait and see," said Nadim.
The children went to Biff's room.
Nadim had the big box. He opened it and took out his computer.
"Brilliant!" said Chip.
"I've got some new computer games," said Nadim. "But we can play with them later. I've got something else to play with."
Nadim had some robots.
"These are great," said Kipper.
"I'm going to get one like this. It's my favourite."
Biff and Chip had a robot, too. They went to fetch it.
"We're going to have a great time," said Nadim.
Wilf came to play. He had one of his new toys. It was a space craft.
Biff, Wilf and Kipper played with all the toys. Kipper said he wanted to be a robot when he grew up.
"You can't," said Biff. "People can't be robots."
Chip and Nadim played on the computer. One of Nadim's new games was called "Storm Castle". It looked exciting.
"It's quite hard," said Nadim. "I'll show you how to play, then you can have a go."
"It looks great," said Chip.
"""
# run_alignment(ORIGINAL_TEXT)
audio_path = "/gdrive/MyDrive/media_proc/speed-align/audio_input/6b_05_Storm_Castle.mp3"
whisper_json_path = "/gdrive/MyDrive/media_proc/speed-align/output/6b_05_Storm_Castle_whisper_segments.json"
output_dir   = "/gdrive/MyDrive/media_proc/speed-align/output"
run_alignment(
        ORIGINAL_TEXT,
        audio_path,
        whisper_json_path,
        output_dir,
        language="en"
    )

🚀 [开始任务] 设备: CPU
📄 [步骤 1/6] 原文切分完成，共 29 句。
   ✅ 数据加载成功，音频时长: 832.37s
🔗 [步骤 3/6] 开始全局序列比对...
📏 [步骤 4/6] 正在对未匹配句子进行时间插值补全...
⏳ [步骤 5/6] 加载 WhisperX 模型执行对齐...

🎯 [开始分块强制对齐] (Max Chunk: 60.0s)
--------------------------------------------------------------------------------
📦 处理 Chunk (索引 0 到 9):
   - 成功捕获 71 个单词
----------------------------------------
📦 处理 Chunk (索引 10 到 23):
   - 成功捕获 103 个单词
----------------------------------------
📦 处理 Chunk (索引 24 到 28):
   - 成功捕获 45 个单词
----------------------------------------
🧩 [步骤 6/6] 正在组装 WhisperX 标准 segments 结构...
      ⚠️ [跳过无效单词]: 句子 0 中的 'unknown' 对齐失败，无时间戳。
      ⚠️ [跳过无效单词]: 句子 0 中的 'unknown' 对齐失败，无时间戳。
      ⚠️ [句子对齐失败]: 第 0 句未捕获到任何有效词级时间戳，已跳过。
      ⚠️ [跳过无效单词]: 句子 1 中的 'unknown' 对齐失败，无时间戳。
      ⚠️ [跳过无效单词]: 句子 1 中的 'unknown' 对齐失败，无时间戳。
      ⚠️ [句子对齐失败]: 第 1 句未捕获到任何有效词级时间戳，已跳过。
      ⚠️ [跳过无效单词]: 句子 2 中的 'unknown' 对齐失败，无时间戳。
      ⚠️ [跳过无效单词]: 句子 2 中的 'unknown' 对齐失败，无时间戳。
      ⚠️ [跳过无效单词]: 句子 2 中的 'unknown' 对齐失败，无时间戳。
      ⚠️ [跳过无效单词]

{'segments': []}

# 双轮驱动

In [ ]:
import sys
sys.path.insert(0, '/content/.venv/lib/python3.12/site-packages')

import os
import re
import json
import torch
import whisperx
import difflib
from typing import List, Dict, Any

# ──────────────────────────────────────────────────────────────────
# 1. 配置与工具函数
# ──────────────────────────────────────────────────────────────────

class AlignConfig:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"
    MIN_SILENCE_GAP = 0.5    # 触发 Chunk 切分的静音阈值 (秒)
    MAX_CHUNK_DURATION = 40.0 # 单个对齐块的最大时长 (秒)
    WORD_GAP_BUFFER = 0.01   # 强制单词间留出的最小物理间隔 (10ms)
    SIM_THRESHOLD = 0.85

def normalize(t: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(t).lower())

def clean_markdown_keep_paragraphs(md_text: str) -> str:
    """
    1. 剔除 ![](images/...) 图片和 Markdown 符号
    2. 去掉多余空行，仅保留段落间的单换行
    3. 如果行尾没有标点且是文本，自动补齐句号
    """
    # 1. 彻底剔除图片链接 ![alt](url) 或 ![](url)
    text = re.sub(r'!\[.*?\]\(.*?\)', '', md_text)

    # 2. 剔除普通超链接 [text](url) -> 仅保留文字
    text = re.sub(r'\[([^\]]+)\]\(.*?\)', r'\1', text)

    # 3. 剔除标题符号 #、加粗 *、斜体 _
    text = re.sub(r'^#+\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'[*_]{1,3}', '', text)

    # 4. 按行处理：去除多余空行，补齐标点
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # 跳过真正的空行
        if not line:
            continue

        # 5. 自动补全标点
        # 如果行末是字母、数字或中文，说明可能是标题或断句，补上句号
        if re.search(r'[a-zA-Z0-9\u4e00-\u9fa5]$', line):
            line += "."

        cleaned_lines.append(line)

    # 6. 用单个换行符连接，保留段落感，但没有多余空行
    return "\n".join(cleaned_lines)

# ──────────────────────────────────────────────────────────────────
# 2. 核心双轨对齐类
# ──────────────────────────────────────────────────────────────────

class AudioBookAligner:
    def __init__(self, audio_path: str, original_text: str, language="en"):
        self.audio = whisperx.load_audio(audio_path)
        self.audio_dur = len(self.audio) / 16000
        self.lang = language
        self.sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', original_text.strip()) if s.strip()]

        self.data_a = []  # A轨: 全局参考坐标 (Reference)
        self.data_b = []  # B轨: 强制对齐结果 (Candidate)

    def load_a_track(self, a_json_path: str):
        """加载已有的全局转写 A 轨数据"""
        print(f"📖 [A轨] 正在加载全局转写坐标系...")
        with open(a_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for seg in data.get("segments", []):
                for w in seg.get("words", []):
                    if w.get("word") and w.get("start") is not None:
                        self.data_a.append({
                            "word": w["word"], "start": w["start"],
                            "end": w["end"], "score": w.get("score", 0.0)
                        })
        print(f"   ✅ A轨就绪: {len(self.data_a)} 词")

    def run_forced_alignment_b(self, chunk_padding=1.2):
        """
        核心重构：基于静音感知的 B 轨强制对齐
        """
        print(f"🎯 [B轨] 开始执行静音感知分块对齐...")

        # 1. 通过 A 轨映射出每句原文的估算区间
        sent_intervals = self._estimate_intervals_via_a()

        # 2. 动态分块：寻找 A 轨中的静音裂缝进行切割
        chunks = self._create_silent_aware_chunks(sent_intervals)

        # 3. 执行对齐
        align_model, metadata = whisperx.load_align_model(language_code=self.lang, device=AlignConfig.DEVICE, model_name="WAV2VEC2_ASR_LARGE_LV60K_960H")

        for c_idx, chunk in enumerate(chunks):
            chunk_text = " ".join([s["text"] for s in chunk["sents"]])
            c_start = max(0, chunk["start_t"] - chunk_padding)
            c_end = min(self.audio_dur, chunk["end_t"] + chunk_padding)

            # 截取音频并对齐
            audio_chunk = self.audio[int(c_start * 16000) : int(c_end * 16000)]
            spec = [{"text": chunk_text, "start": 0.0, "end": len(audio_chunk)/16000}]

            try:
                res = whisperx.align(spec, align_model, metadata, audio_chunk, AlignConfig.DEVICE)
                chunk_words = []
                for s in res["segments"]:
                    for w in s.get("words", []):
                        if w.get("start") is not None:
                            chunk_words.append({
                                "word": w["word"],
                                "start": round(w["start"] + c_start, 3),
                                "end": round(w["end"] + c_start, 3),
                                "score": w.get("score", 0.0)
                            })

                # 4. 时间轴平滑：修正 Chunk 内部单词重叠
                # self._smooth_timestamps(chunk_words)
                self.data_b.extend(chunk_words)

            except Exception as e:
                print(f"   ⚠️ Chunk {c_idx} 对齐失败: {e}")

        print(f"   ✅ B轨分块对齐完成，共捕获 {len(self.data_b)} 词。")

    def _create_silent_aware_chunks(self, sent_intervals):
        """利用 A 轨停顿信息将句子组合成最优 Chunk"""
        chunks = []
        current_chunk = {"sents": [], "start_t": None, "end_t": None}

        for i, (sent, interval) in enumerate(zip(self.sentences, sent_intervals)):
            s_t, e_t = interval["start"], interval["end"]
            if current_chunk["start_t"] is None: current_chunk["start_t"] = s_t

            # 切分判据：1. 时长超限 2. 探测到 A 轨中的静音 Gap
            should_split = False
            if (e_t - current_chunk["start_t"]) > AlignConfig.MAX_CHUNK_DURATION:
                should_split = True
            elif i < len(sent_intervals) - 1:
                next_s_t = sent_intervals[i+1]["start"]
                if next_s_t and e_t and (next_s_t - e_t) > AlignConfig.MIN_SILENCE_GAP:
                    should_split = True # 在这个天然静音处切断

            current_chunk["sents"].append({"text": sent, "interval": interval})
            current_chunk["end_t"] = e_t

            if should_split or i == len(self.sentences) - 1:
                chunks.append(current_chunk)
                current_chunk = {"sents": [], "start_t": None, "end_t": None}
        return chunks

    def _estimate_intervals_via_a(self):
        """核心：通过 difflib 将原文单词与 A 轨单词流全局最优对齐"""
        origin_words = [normalize(w) for sent in self.sentences for w in sent.split() if normalize(w)]
        a_words_norm = [normalize(x["word"]) for x in self.data_a]

        matcher = difflib.SequenceMatcher(None, origin_words, a_words_norm)
        # 获取每句对应的原文单词起止索引
        indices = []
        curr = 0
        for sent in self.sentences:
            count = len([w for w in sent.split() if normalize(w)])
            indices.append((curr, curr + count))
            curr += count

        intervals = []
        opcodes = matcher.get_opcodes()
        for start_i, end_i in indices:
            t_s, t_e = None, None
            for tag, i1, i2, j1, j2 in opcodes:
                if tag == 'equal':
                    s, e = max(start_i, i1), min(end_i, i2)
                    if s < e:
                        if t_s is None: t_s = self.data_a[j1 + (s - i1)]["start"]
                        t_e = self.data_a[j1 + (e - i1) - 1]["end"]
            intervals.append({"start": t_s, "end": t_e})

        # 简单线性插值补全缺失
        self._interpolate_intervals(intervals)
        return intervals

    def _interpolate_intervals(self, intervals):
        last_e = 0.0
        for i in range(len(intervals)):
            if intervals[i]["start"] is None:
                nxt_s = next((intervals[j]["start"] for j in range(i+1, len(intervals)) if intervals[j]["start"]), self.audio_dur)
                intervals[i]["start"], intervals[i]["end"] = last_e, (last_e + nxt_s) / 2
            last_e = intervals[i]["end"]

    def _smooth_timestamps(self, words):
        """修正单词重叠：前词末尾必须在后词开头之前"""
        for i in range(1, len(words)):
            prev_e = words[i-1]["end"]
            curr_s = words[i]["start"]
            if curr_s < prev_e + AlignConfig.WORD_GAP_BUFFER:
                # 修正：将当前词起点推后，确保不重叠
                words[i]["start"] = round(prev_e + AlignConfig.WORD_GAP_BUFFER, 3)
                # 联动修正：确保单词长度至少有 50ms
                if words[i]["end"] < words[i]["start"]:
                    words[i]["end"] = round(words[i]["start"] + 0.05, 3)

    def merge_and_save(self, output_path: str):
        """最终融合：B 轨主打，A 轨填坑"""
        print(f"🧩 [阶段 3] 正在融合输出最终结果...")
        ptr_a, ptr_b = 0, 0
        final_segments = []

        for i, sent_text in enumerate(self.sentences):
            tokens = sent_text.split()
            matched_words = []

            for token in tokens:
                target = normalize(token)
                if not target: continue

                # 搜索 B 轨，搜索不到则搜 A 轨
                found = self._find_in_pool(target, self.data_b, ptr_b)
                if found:
                    match, ptr_b = found
                    matched_words.append({
                        "word": token, "start": match["start"],
                        "end": match["end"], "score": match["score"], "m": "forced_b"
                    })
                else:
                    found = self._find_in_pool(target, self.data_a, ptr_a)
                    if found:
                        match, ptr_a = found
                        matched_words.append({
                            "word": token, "start": match["start"],
                            "end": match["end"], "score": match["score"], "m": "fallback_a"
                        })
                    else:
                        matched_words.append({
                            "word": token, "start": None, "end": None, "score": 0, "m": "MANUAL"
                        })

            # 封装 Segment
            v = [w["start"] for w in matched_words if w["start"] is not None]
            final_segments.append({
                "text": sent_text,
                "start": v[0] if v else None,
                "end": [w["end"] for w in matched_words if w["end"] is not None][-1] if v else None,
                "words": matched_words
            })

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump({"segments": final_segments}, f, indent=2, ensure_ascii=False)
        print(f"🎉 处理完成！文件保存至: {output_path}")

    def _find_in_pool(self, target, pool, ptr):
        for i in range(ptr, min(ptr + 20, len(pool))):
            if difflib.SequenceMatcher(None, target, normalize(pool[i].get("word") or pool[i].get("w"))).ratio() > AlignConfig.SIM_THRESHOLD:
                return pool[i], i + 1
        return None

# ──────────────────────────────────────────────────────────────────
# 3. 使用入口
# ──────────────────────────────────────────────────────────────────
# ORIGINAL_TEXT = """..."""
# aligner = AudioBookAligner("audio.mp3", ORIGINAL_TEXT)
# aligner.load_a_track("whisper_a.json")
# aligner.run_forced_alignment_b()
# aligner.merge_and_save("final_result.json")

# 这里填入你的文件路径
ORIGINAL_TEXT = """
# Lesson 5

# Storm Castle

![](images/89d01bd6a0b8b45cf0c2b6a761eaf865b90fdd9b2966007ee6e46e0732501a1d.jpg)


![](images/94e3b08e8527137dae5ce651bda3aa93541e44c4c7c937d80f80a58403d0e8fe.jpg)


Nadim's mum and dad had to go away, so Nadim came to stay at Biff and Chip's house.

Nadim had a bag with all his things, but he had a big box, too.

"What's in that big box?" asked Chip.

"Wait and see," said Nadim.


![](images/11a0b377f267d156b3a7615239fb8fa23e630ee140a0f850aa6c1faaaecced6d.jpg)


The children went to Biff's room.

Nadim had the big box. He opened it and took out his computer.

"Brilliant!" said Chip.

"I've got some new computer games," said Nadim. "But we can play with them later. I've got something else to play with."


![](images/2018f8b806f7a906ff613d0a78d67702b54b5dc44bf15f3609b46785adfafd06.jpg)


Nadim had some robots.

"These are great," said Kipper.

"I'm going to get one like this. It's my favourite."

Biff and Chip had a robot, too. They went to fetch it.

"We're going to have a great time," said Nadim.

![](images/4a63f8818532915caf88555dad5e00a003c3e73226856c32da56c80c7018e1b4.jpg)


Wilf came to play. He had one of his new toys. It was a space craft.

Biff, Wilf and Kipper played with all the toys. Kipper said he wanted to be a robot when he grew up.

"You can't," said Biff. "People can't be robots."

![](images/de2c316ad09e5e6ede47cc6e6499e91b5cf7f5daa50ef035449983e359ea5dc7.jpg)


Chip and Nadim played on the computer. One of Nadim's new games was called "Storm Castle". It looked exciting.

"It's quite hard," said Nadim. "I'll show you how to play, then you can have a go."

"It looks great," said Chip.

![](images/74f82ece7dbee941d2c4844a0b310205add7dfef128530327b56faeb23f0daa3.jpg)


Everyone watched Nadim play Storm Castle.

"You have to go through all the rooms," he said. "But there is a danger in every room. Look."

In the first room, the floor opened. You could fall through, but Nadim didn't.

"That was clever," said Chip.

Nadim was good on the computer. He got through all the rooms safely.

"Chip can have a go next," said Nadim. "You can all have a go if you like."

"You're brilliant at it," said Chip. "I won't be very good when I have my go."

![](images/bd0198310b0d769ca3743a57ddedc2ba7d263133aeca2b732eb28a6c74a4d566.jpg)


Suddenly, the magic key began to glow.

"Oh no!" said Biff. "I don't want the key to glow. I don't want a magic adventure. I want to play on Nadim's computer."

"I don't want a magic adventure, either," said Chip. "I think I know where it will be."

![](images/b0c4a090bffb235bed14cfe981e60339eb6cb3efc69f8ea5160335325e7ff154.jpg)


"Where do you think it will be?" asked Kipper.

"Storm Castle," said Chip.

"Oh no!" said Wilf. "I don't think we're going to like this adventure. Storm Castle is full of dangers."

"It's a good job Nadim is with us," said Biff.

![](images/5fa76cdd280ba920d9f40c99fe4e448a8efcc1f3a9c6a9ed41ea20de7d2ea162.jpg)

The magic didn't take them inside the castle. It took them to a desert. Storm Castle was in front of them.

"Why didn't the magic take us inside the castle?" said Wilf.

"And where's Nadim?" asked Chip.

![](images/d0ae70bf2d6248ec980ad4c54a0adca1f7f8bd5d842e7d3f72773b9c4c51313a.jpg)

"Oh no! Giant robots," said Wilf. "Run for it!"
"""
audio_path = "/gdrive/MyDrive/media_proc/speed-align/audio_input/6b_05_Storm_Castle.mp3"
whisper_json_path = "/gdrive/MyDrive/media_proc/speed-align/output/6b_05_Storm_Castle.json"
OUTPUT_DIR = "/gdrive/MyDrive/media_proc/speed-align/output/"
aligner = AudioBookAligner(audio_path, clean_markdown_keep_paragraphs(ORIGINAL_TEXT))
aligner.load_a_track(whisper_json_path)
aligner.run_forced_alignment_b()
aligner.merge_and_save(OUTPUT_DIR + "final_result.json")


📖 [A轨] 正在加载全局转写坐标系...
   ✅ A轨就绪: 1387 词
🎯 [B轨] 开始执行静音感知分块对齐...
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:01<00:00, 315MB/s]


   ✅ B轨分块对齐完成，共捕获 444 词。
🧩 [阶段 3] 正在融合输出最终结果...
🎉 处理完成！文件保存至: /gdrive/MyDrive/media_proc/speed-align/output/final_result.json
